In [5]:
import requests
import os
from langchain_groq import ChatGroq

from load_dotenv import load_dotenv

# this function load all the variables from the .env file and 
# will make them available in the os.environ dictionary (env variable)
load_dotenv()

llm_groq = ChatGroq(model="meta-llama/llama-prompt-guard-2-22m", temperature=0)


if os.environ.get("GROQ_API_KEY"):
    print("got the api key")
else:
    print("not got api key")

got the api key


In [2]:
import requests
from langchain_ollama import ChatOllama

# Ollama typically runs on localhost:11434
ollama_url = "http://localhost:11434"

try:
    # Send a request to check if the local Ollama server is active
    response = requests.get(ollama_url)
    if response.status_code == 200:
        print("Ollama server is running locally!")
        
        # Initialize the model (replace 'llama3' with your downloaded model)
        llm = ChatOllama(model="qwen2.5:3b", base_url=ollama_url)
        print("ChatOllama initialized successfully.")
    else:
        print(f"Ollama server responded with status code: {response.status_code}")
except requests.exceptions.ConnectionError:
    print("Ollama server is not running. Please start the Ollama application.")


Ollama server is running locally!
ChatOllama initialized successfully.


In [3]:
from langchain_ollama import ChatOllama

llm_ollama= ChatOllama(model='qwen2.5:3b', temperature=0)

In [1]:
# from openai.types import model
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate

# it only give the content part of the output (task3)
from langchain_core.output_parsers import StrOutputParser, PydanticOutputParser

llm_ollama= ChatOllama(model='qwen2.5:3b', temperature=0)
my_messages = [
    SystemMessage(content="you are a gen-z assistant who always answer in fun way"),
    HumanMessage(content="hey")
]

llm_ollama.invoke(my_messages).content

'Hey there! It\'s like saying "hello" but with extra enthusiasm and a dash of fun. How’s your day so far? Ready for some laughs or maybe we can play a quick game?'

In [2]:
'''
we need to use this particular thing in our downstream models as well .. but this is a pydantic model
how we can just parse it ??
you have two options -->>
1... you can either create your own custom function custom runnabble to use it
2... other one thing is provided by langchain is called PydanticOutputParser (in above block line 7) it will simply parse it    

'''


from pydantic import BaseModel
from typing import Literal

# Literal means you have only this much of option you have to answer in these options only

class llm_schema(BaseModel):
    movie_summary_flag: Literal['positive','negative']

llm_structure_output = llm_ollama.with_structured_output(llm_schema)

# test  the schema 
llm_structure_output.invoke("the movie is great")

llm_schema(movie_summary_flag='positive')

# **Chain with Conditional Chains**

In [3]:
# prompt 1st TASK

prompt_template = ChatPromptTemplate([
    ('system','You are a movie review evaluator'),
    ('human','Please categorize movie review as positive or negative : {input}')
])

In [6]:
# TASK 2 LLM

llm_groq = ChatGroq(model="meta-llama/llama-prompt-guard-2-22m", temperature=0)
llm_ollama= ChatOllama(model='qwen2.5:3b', temperature=0)


In [7]:
# TASK 3 custom runnable

from langchain_core.runnables import RunnableLambda

def dict_maker(text: str)-> dict :
    return {"text": text}

dict_maker_runnable= RunnableLambda(dict_maker)

## **Parallel Chain 1**


In [13]:
# TASK 1 PROMPT

LinkedIn_post = ChatPromptTemplate.from_messages([
    ('system', 'You are a linkedIn post generator'),
    ('human', 'Create a post for the following text for LinkedIn : {text}')
])

# TASK 2 LLM

llm_groq = ChatGroq(model="meta-llama/llama-prompt-guard-2-22m", temperature=0)
llm_ollama= ChatOllama(model='qwen2.5:3b', temperature=0)

# TASK 3 Str PARSER

parcer = StrOutputParser()

chain_linkedIn = LinkedIn_post | llm_ollama | parcer

## **Parallel Chain 2**

In [15]:
from langchain_core.runnables import RunnableParallel

In [16]:
def insta_chain(text : dict):

    text = text['text']
    # TASK 1 PROMPT

    Insta_prompt = ChatPromptTemplate.from_messages([
        ('system', 'You are a Instagram post generator'),
        ('human', 'Create a post for the following text for Instagram : {text}')
    ])

    # TASK 2 LLM

    llm_groq = ChatGroq(model="meta-llama/llama-prompt-guard-2-22m", temperature=0)
    llm_ollama= ChatOllama(model='qwen2.5:3b', temperature=0)

    # TASK 3 Str PARSER

    parcer = StrOutputParser()

    chain_insta = Insta_prompt |llm_ollama | parcer

    result = chain_insta.invoke(text)

    return result

insta_chain_runnable = RunnableLambda(insta_chain)

## **Final Orchestration**

In [17]:
final_chain = (
    prompt_template |
    llm_ollama |
    parcer |
    dict_maker_runnable |
    RunnableParallel(branches={ 'linkedIn': chain_linkedIn, 'Instagram': insta_chain_runnable})
)

In [18]:
final_chain.invoke('iron man')

{'branches': {'linkedIn': 'Here’s a LinkedIn post based on the provided text:\n\n---\n\nLinkedIn Community,\n\nI don’t have direct access to specific reviews of "Iron Man," but generally speaking, films like Iron Man within the Marvel Cinematic Universe tend to receive positive feedback. Reviews often highlight impressive action sequences, well-developed characters, and stellar performances as key strengths.\n\nWhile I can\'t definitively evaluate this particular film\'s reception without a review in hand, based on typical audience expectations, one would expect "Iron Man" to be viewed positively overall. \n\nWhat are your thoughts on the Marvel Cinematic Universe? Have you seen Iron Man or any other MCU films and what did you think?\n\n#MarvelCinematicUniverse #IronMan #MovieReviews\n\n---\n\nFeel free to adjust any details as needed for a more personalized post!',
  'Instagram': 'Here’s a polished version of your text for an Instagram post:\n\n🎬 **"Iron Man" is often praised in the M

## **Chain as a runnable**

In [21]:
# TASK 1 BUFITY FUNCTION

def beautify(final_response: dict)->dict :
    linkedIn_response= final_response['branches']['linkedIn']
    instagram_response= final_response['branches']['Instagram']

    return {'LINKEDIN': linkedIn_response, 'INSTAGRAM': instagram_response}

beautify_runnable = RunnableLambda(beautify)

# TASK 2 FINAL CHAIN

# final_chain the product of a chain is a runnable you dont need to make it runnable as to a function

# beaytified chain

beautify_chain = final_chain | beautify_runnable 

beautify_chain.invoke('avengers:endgame')

{'LINKEDIN': 'Here’s a LinkedIn post based on your provided text:\n\n---\n\nInaccurate sentiment analysis of movie reviews can lead to misinterpretation. To truly understand the sentiment behind a review for "Avengers: Endgame," it\'s essential to read and analyze that specific piece of writing. Movie reviews vary widely in tone, from enthusiastic praise to critical analysis.\n\nWithout access to the actual text, determining whether a review is positive or negative is impossible. If you have a particular review in mind or would like me to discuss "Avengers: Endgame" in general terms, I\'d be delighted to do so! 🎬🌟\n\n---',
 'INSTAGRAM': "📸 **Caption:** 📖 Dive into the heart of movie reviews! To accurately gauge how fans felt about Avengers: Endgame, I need to read and analyze their specific thoughts. Movie reviews can swing from glowing praise to biting critiques. But without seeing the actual words, it's like trying to guess a puzzle with missing pieces. 🎬💭 #MovieReviews #AvengersEndg